In [15]:
import os
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    balanced_accuracy_score, roc_auc_score, average_precision_score,
    matthews_corrcoef, confusion_matrix, precision_recall_curve
)

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, models

# ----------------------------
# Main settings
# ----------------------------
DATA_DIR = r"E:\DL-Projects\EL-DSLR-Benchmarking\Cells-labelled"
RESULTS_DIR = Path("results_final_benchmark")

CLASS_NAMES = ["Normal", "Defective"]
IMG_SIZE = 224
WEIGHT_DECAY = 1e-4

BASE_SEED = 42
N_SPLITS = 3
NUM_EPOCHS = 80
PATIENCE = 12
BATCH_SIZE = 32


for sub in [
    "metrics", "models", "predictions", "confusion_matrices",
    "splits", "misclassified", "complexity"
]:
    (RESULTS_DIR / sub).mkdir(parents=True, exist_ok=True)



device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

for sub in ["metrics", "models", "predictions", "confusion_matrices", "splits"]:
    (RESULTS_DIR / sub).mkdir(parents=True, exist_ok=True)

Device: cuda


In [16]:
def collect_image_paths(data_dir):
    rows = []
    data_dir = Path(data_dir)

    for class_name in CLASS_NAMES:
        class_dir = data_dir / class_name

        if not class_dir.exists():
            raise FileNotFoundError(f"Missing folder: {class_dir}")

        for img_path in class_dir.glob("*"):
            if img_path.suffix.lower() in [".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"]:
                rows.append({
                    "path": str(img_path),
                    "filename": img_path.name,
                    "label_name": class_name,
                    "label": CLASS_NAMES.index(class_name)
                })

    df = pd.DataFrame(rows)
    print(df["label_name"].value_counts())
    print("Total images:", len(df))
    return df


df = collect_image_paths(DATA_DIR)
df.head()

label_name
Normal       125
Defective     19
Name: count, dtype: int64
Total images: 144


,path,filename,label_name,label
0,E:\DL-Projects\EL-DSLR-Benchmarking\Cells-labe...,cell_10.png,Normal,0
1,E:\DL-Projects\EL-DSLR-Benchmarking\Cells-labe...,cell_100.png,Normal,0
2,E:\DL-Projects\EL-DSLR-Benchmarking\Cells-labe...,cell_101.png,Normal,0
3,E:\DL-Projects\EL-DSLR-Benchmarking\Cells-labe...,cell_102.png,Normal,0
4,E:\DL-Projects\EL-DSLR-Benchmarking\Cells-labe...,cell_103.png,Normal,0


In [17]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [18]:
train_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=8),
    transforms.RandomAffine(
        degrees=0,
        translate=(0.015, 0.015),
        scale=(0.98, 1.02)
    ),
    transforms.ColorJitter(
        brightness=0.08,
        contrast=0.10
    ),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

eval_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])


class ELCellDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["path"]).convert("RGB")
        label = int(row["label"])

        if self.transform:
            img = self.transform(img)

        return img, label, row["path"]

In [19]:
def calculate_metrics(y_true, y_pred, y_prob=None):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    out = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "specificity": tn / (tn + fp) if (tn + fp) > 0 else 0,
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "mcc": matthews_corrcoef(y_true, y_pred),
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp
    }

    if y_prob is not None and len(np.unique(y_true)) == 2:
        out["roc_auc"] = roc_auc_score(y_true, y_prob)
        out["pr_auc"] = average_precision_score(y_true, y_prob)
    else:
        out["roc_auc"] = np.nan
        out["pr_auc"] = np.nan

    return out


def save_confusion_matrix(y_true, y_pred, title, save_path):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])

    plt.figure(figsize=(4, 4))
    plt.imshow(cm)
    plt.title(title)
    plt.xticks([0, 1], CLASS_NAMES, rotation=45)
    plt.yticks([0, 1], CLASS_NAMES)
    plt.xlabel("Predicted")
    plt.ylabel("True")

    for i in range(2):
        for j in range(2):
            plt.text(j, i, str(cm[i, j]), ha="center", va="center")

    plt.tight_layout()
    plt.savefig(save_path, dpi=300)
    plt.close()

In [20]:
def find_best_threshold(y_true, y_prob, mode="f1", min_recall=None):
    precision, recall, thresholds = precision_recall_curve(y_true, y_prob)

    precision = precision[:-1]
    recall = recall[:-1]

    if mode == "f1":
        scores = 2 * precision * recall / (precision + recall + 1e-8)
    elif mode == "f2":
        scores = 5 * precision * recall / (4 * precision + recall + 1e-8)
    else:
        raise ValueError("mode must be 'f1' or 'f2'")

    if min_recall is not None:
        valid = recall >= min_recall
        if valid.sum() > 0:
            valid_scores = scores.copy()
            valid_scores[~valid] = -1
            best_idx = np.argmax(valid_scores)
        else:
            best_idx = np.argmax(scores)
    else:
        best_idx = np.argmax(scores)

    return {
        "threshold": float(thresholds[best_idx]),
        "val_precision": float(precision[best_idx]),
        "val_recall": float(recall[best_idx]),
        "val_score": float(scores[best_idx]),
        "threshold_mode": mode
    }


def apply_threshold(y_prob, threshold):
    return (np.array(y_prob) >= threshold).astype(int)

In [21]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, reduction="mean"):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, logits, targets):
        ce_loss = nn.functional.cross_entropy(
            logits,
            targets,
            weight=self.alpha,
            reduction="none"
        )
        pt = torch.exp(-ce_loss)
        loss = ((1 - pt) ** self.gamma) * ce_loss

        if self.reduction == "mean":
            return loss.mean()
        elif self.reduction == "sum":
            return loss.sum()
        return loss


def get_class_weights(train_df):
    labels = train_df["label"].values
    counts = np.bincount(labels)
    total = counts.sum()
    weights = total / (len(counts) * counts)
    return torch.tensor(weights, dtype=torch.float32).to(device)


def make_weighted_sampler(train_df):
    labels = train_df["label"].values
    counts = np.bincount(labels)
    class_weights = 1.0 / counts
    sample_weights = class_weights[labels]

    return WeightedRandomSampler(
        weights=torch.DoubleTensor(sample_weights),
        num_samples=len(sample_weights),
        replacement=True
    )

In [22]:
def build_model(model_name, num_classes=2):
    model_name = model_name.lower()

    if model_name == "resnet18":
        weights = models.ResNet18_Weights.DEFAULT
        model = models.resnet18(weights=weights)
        model.fc = nn.Linear(model.fc.in_features, num_classes)

    elif model_name == "mobilenetv2":
        weights = models.MobileNet_V2_Weights.DEFAULT
        model = models.mobilenet_v2(weights=weights)
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)

    elif model_name == "efficientnet_b0":
        weights = models.EfficientNet_B0_Weights.DEFAULT
        model = models.efficientnet_b0(weights=weights)
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
        
        
    elif model_name == "resnet34":
        weights = models.ResNet34_Weights.DEFAULT
        model = models.resnet34(weights=weights)
        model.fc = nn.Linear(model.fc.in_features, num_classes)

    elif model_name == "densenet121":
        weights = models.DenseNet121_Weights.DEFAULT
        model = models.densenet121(weights=weights)
        model.classifier = nn.Linear(model.classifier.in_features, num_classes)

    elif model_name == "shufflenetv2":
        weights = models.ShuffleNet_V2_X1_0_Weights.DEFAULT
        model = models.shufflenet_v2_x1_0(weights=weights)
        model.fc = nn.Linear(model.fc.in_features, num_classes)

    else:
        raise ValueError(f"Unknown model name: {model_name}")

    return model


def set_backbone_trainable(model, model_name, trainable):
    model_name = model_name.lower()

    for param in model.parameters():
        param.requires_grad = trainable

    # Always keep classifier trainable
    if model_name == "resnet18":
        for param in model.fc.parameters():
            param.requires_grad = True
    elif model_name == "mobilenetv2":
        for param in model.classifier.parameters():
            param.requires_grad = True
    elif model_name == "efficientnet_b0":
        for param in model.classifier.parameters():
            param.requires_grad = True
            
    elif model_name == "resnet34":
        for param in model.fc.parameters():
            param.requires_grad = True

    elif model_name == "densenet121":
        for param in model.classifier.parameters():
            param.requires_grad = True
        
    elif model_name == "shufflenetv2":
        for param in model.fc.parameters():
            param.requires_grad = True

In [23]:
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    running_loss = 0.0
    total = 0

    for images, labels, _ in loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        total += labels.size(0)

    return running_loss / total


def evaluate_model(model, loader, criterion=None):
    model.eval()

    y_true, y_prob, paths = [], [], []
    running_loss = 0.0
    total = 0

    with torch.no_grad():
        for images, labels, batch_paths in loader:
            images = images.to(device)
            labels = labels.to(device)

            logits = model(images)
            probs = torch.softmax(logits, dim=1)[:, 1]

            if criterion is not None:
                loss = criterion(logits, labels)
                running_loss += loss.item() * images.size(0)
                total += labels.size(0)

            y_true.extend(labels.cpu().numpy())
            y_prob.extend(probs.cpu().numpy())
            paths.extend(batch_paths)

    avg_loss = running_loss / total if criterion is not None else None

    return np.array(y_true), np.array(y_prob), paths, avg_loss

In [24]:
def run_one_fold_config(
    df,
    train_idx,
    test_idx,
    fold,
    config,
    base_seed=42
):
    seed = base_seed + fold
    set_seed(seed)

    trainval_df = df.iloc[train_idx].reset_index(drop=True)
    test_df = df.iloc[test_idx].reset_index(drop=True)

    train_df, val_df = train_test_split(
        trainval_df,
        test_size=0.20,
        stratify=trainval_df["label"],
        random_state=seed
    )

    train_df = train_df.reset_index(drop=True)
    val_df = val_df.reset_index(drop=True)

    model_name = config["model_name"]
    loss_name = config["loss_name"]
    lr = config["lr"]
    threshold_mode = config["threshold_mode"]
    freeze_epochs = config.get("freeze_epochs", 0)
    use_sampler = config.get("use_sampler", True)

    config_name = (
        f"{model_name}_{loss_name}_lr{lr}_"
        f"{threshold_mode}_freeze{freeze_epochs}_fold{fold}"
    )

    print("\n" + "=" * 80)
    print(config_name)
    print("=" * 80)

    # Save split
    split_dir = RESULTS_DIR / "splits" / config_name
    split_dir.mkdir(parents=True, exist_ok=True)
    train_df.to_csv(split_dir / "train.csv", index=False)
    val_df.to_csv(split_dir / "val.csv", index=False)
    test_df.to_csv(split_dir / "test.csv", index=False)

    train_ds = ELCellDataset(train_df, transform=train_transform)
    val_ds = ELCellDataset(val_df, transform=eval_transform)
    test_ds = ELCellDataset(test_df, transform=eval_transform)

    if use_sampler:
        sampler = make_weighted_sampler(train_df)
        train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,pin_memory=True,  num_workers=0)
    else:
        train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,pin_memory=True, num_workers=0)

    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, pin_memory=True, num_workers=0)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, pin_memory=True, num_workers=0)

    model = build_model(model_name).to(device)

    class_weights = get_class_weights(train_df)

    if loss_name == "weighted_ce":
        criterion = nn.CrossEntropyLoss(weight=class_weights)
    elif loss_name == "focal":
        criterion = FocalLoss(alpha=class_weights, gamma=2.0)
    else:
        criterion = nn.CrossEntropyLoss()

    # Freeze backbone initially if requested
    if freeze_epochs > 0:
        set_backbone_trainable(model, model_name, trainable=False)

    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr,
        weight_decay=WEIGHT_DECAY
    )

    best_val_score = -1.0
    best_path = RESULTS_DIR / "models" / f"{config_name}_best.pth"
    patience_counter = 0
    history = []

    for epoch in range(NUM_EPOCHS):

        # Unfreeze after freeze_epochs
        if epoch == freeze_epochs and freeze_epochs > 0:
            set_backbone_trainable(model, model_name, trainable=True)
            optimizer = torch.optim.AdamW(
                model.parameters(),
                lr=lr,
                weight_decay=WEIGHT_DECAY
            )

        train_loss = train_one_epoch(model, train_loader, optimizer, criterion)

        y_val, prob_val, _, val_loss = evaluate_model(model, val_loader, criterion)

        threshold_info = find_best_threshold(
            y_val,
            prob_val,
            mode=threshold_mode,
            min_recall=None
        )

        val_pred = apply_threshold(prob_val, threshold_info["threshold"])
        val_metrics = calculate_metrics(y_val, val_pred, prob_val)

        # selection score
        if threshold_mode == "f1":
            selection_score = val_metrics["f1"]
        else:
            p = val_metrics["precision"]
            r = val_metrics["recall"]
            selection_score = 5 * p * r / (4 * p + r + 1e-8)

        history.append({
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "threshold": threshold_info["threshold"],
            "val_accuracy": val_metrics["accuracy"],
            "val_precision": val_metrics["precision"],
            "val_recall": val_metrics["recall"],
            "val_f1": val_metrics["f1"],
            "val_balanced_accuracy": val_metrics["balanced_accuracy"],
            "val_pr_auc": val_metrics["pr_auc"],
            "selection_score": selection_score
        })

        print(
            f"Epoch {epoch+1:03d} | "
            f"Loss {train_loss:.4f} | "
            f"Val F1 {val_metrics['f1']:.4f} | "
            f"Val Recall {val_metrics['recall']:.4f} | "
            f"Thr {threshold_info['threshold']:.3f}"
        )

        if selection_score > best_val_score:
            best_val_score = selection_score
            torch.save(model.state_dict(), best_path)
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= PATIENCE:
            print(f"Early stopping at epoch {epoch+1}")
            break

    pd.DataFrame(history).to_csv(
        RESULTS_DIR / "metrics" / f"{config_name}_history.csv",
        index=False
    )

    # Load best checkpoint
    model.load_state_dict(torch.load(best_path, map_location=device))

    # Recompute best threshold on validation set
    y_val, prob_val, _, _ = evaluate_model(model, val_loader)
    threshold_info = find_best_threshold(
        y_val,
        prob_val,
        mode=threshold_mode,
        min_recall=None
    )

    best_threshold = threshold_info["threshold"]

    # Final test fold evaluation
    y_test, prob_test, paths, _ = evaluate_model(model, test_loader)
    y_pred = apply_threshold(prob_test, best_threshold)

    metrics = calculate_metrics(y_test, y_pred, prob_test)

    metrics.update({
        "fold": fold,
        "model": model_name,
        "loss": loss_name,
        "lr": lr,
        "threshold_mode": threshold_mode,
        "threshold": best_threshold,
        "freeze_epochs": freeze_epochs,
        "use_sampler": use_sampler,
        "config_name": config_name
    })

    pred_df = pd.DataFrame({
        "path": paths,
        "y_true": y_test,
        "y_pred": y_pred,
        "prob_defective": prob_test,
        "threshold": best_threshold
    })

    pred_df["true_name"] = pred_df["y_true"].map({0: "Normal", 1: "Defective"})
    pred_df["pred_name"] = pred_df["y_pred"].map({0: "Normal", 1: "Defective"})

    pred_df.to_csv(
        RESULTS_DIR / "predictions" / f"{config_name}_predictions.csv",
        index=False
    )

    save_confusion_matrix(
        y_test,
        y_pred,
        config_name,
        RESULTS_DIR / "confusion_matrices" / f"{config_name}_cm.png"
    )

    return metrics

In [25]:
FINAL_DEEP_CONFIGS = [
    {
        "model_name": "mobilenetv2",
        "loss_name": "focal",
        "lr": 1e-4,
        "threshold_mode": "f2",
        "freeze_epochs": 5,
        "use_sampler": True
    },
    {
        "model_name": "resnet18",
        "loss_name": "weighted_ce",
        "lr": 1e-4,
        "threshold_mode": "f2",
        "freeze_epochs": 5,
        "use_sampler": True
    },
    {
        "model_name": "resnet34",
        "loss_name": "weighted_ce",
        "lr": 1e-4,
        "threshold_mode": "f2",
        "freeze_epochs": 0,
        "use_sampler": True
    },
    {
        "model_name": "densenet121",
        "loss_name": "focal",
        "lr": 1e-4,
        "threshold_mode": "f2",
        "freeze_epochs": 0,
        "use_sampler": True
    }
]

In [26]:
from skimage.feature import hog, local_binary_pattern
from skimage.color import rgb2gray
from skimage.transform import resize
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

def extract_features_from_image(path, size=128):
    img = Image.open(path).convert("RGB")
    img = np.array(img)
    img = resize(img, (size, size), anti_aliasing=True)

    gray = rgb2gray(img)

    hog_feat = hog(
        gray,
        orientations=9,
        pixels_per_cell=(16, 16),
        cells_per_block=(2, 2),
        block_norm="L2-Hys",
        feature_vector=True
    )

    lbp = local_binary_pattern(gray, P=8, R=1, method="uniform")
    lbp_hist, _ = np.histogram(
        lbp.ravel(),
        bins=np.arange(0, 11),
        density=True
    )

    stats = np.array([
        gray.mean(),
        gray.std(),
        gray.min(),
        gray.max(),
        np.percentile(gray, 25),
        np.percentile(gray, 50),
        np.percentile(gray, 75)
    ])

    return np.concatenate([hog_feat, lbp_hist, stats])


def build_feature_matrix(split_df):
    X = []
    y = split_df["label"].values

    for path in split_df["path"]:
        X.append(extract_features_from_image(path))

    return np.array(X), y

In [27]:
def run_classical_fold(df, train_idx, test_idx, fold, model_name, base_seed=42):
    seed = base_seed + fold
    set_seed(seed)

    trainval_df = df.iloc[train_idx].reset_index(drop=True)
    test_df = df.iloc[test_idx].reset_index(drop=True)

    train_df, val_df = train_test_split(
        trainval_df,
        test_size=0.20,
        stratify=trainval_df["label"],
        random_state=seed
    )

    X_train, y_train = build_feature_matrix(train_df)
    X_val, y_val = build_feature_matrix(val_df)
    X_test, y_test = build_feature_matrix(test_df)

    if model_name == "SVM":
        clf = Pipeline([
            ("scaler", StandardScaler()),
            ("svm", SVC(
                kernel="rbf",
                C=10,
                gamma="scale",
                class_weight="balanced",
                probability=True,
                random_state=seed
            ))
        ])

    elif model_name == "RandomForest":
        clf = RandomForestClassifier(
            n_estimators=500,
            max_depth=None,
            min_samples_leaf=1,
            class_weight="balanced_subsample",
            random_state=seed,
            n_jobs=-1
        )
    else:
        raise ValueError(model_name)

    clf.fit(X_train, y_train)

    val_prob = clf.predict_proba(X_val)[:, 1]
    threshold_info = find_best_threshold(
        y_val,
        val_prob,
        mode="f2",
        min_recall=None
    )
    threshold = threshold_info["threshold"]

    test_prob = clf.predict_proba(X_test)[:, 1]
    test_pred = apply_threshold(test_prob, threshold)

    metrics = calculate_metrics(y_test, test_pred, test_prob)

    config_name = f"{model_name}_classical_f2_fold{fold}"

    metrics.update({
        "fold": fold,
        "model": model_name,
        "loss": "classical",
        "lr": np.nan,
        "threshold_mode": "f2",
        "threshold": threshold,
        "freeze_epochs": np.nan,
        "use_sampler": False,
        "config_name": config_name
    })

    pred_df = test_df.copy()
    pred_df["y_true"] = y_test
    pred_df["y_pred"] = test_pred
    pred_df["prob_defective"] = test_prob
    pred_df["threshold"] = threshold
    pred_df["true_name"] = pred_df["y_true"].map({0: "Normal", 1: "Defective"})
    pred_df["pred_name"] = pred_df["y_pred"].map({0: "Normal", 1: "Defective"})

    pred_df.to_csv(
        RESULTS_DIR / "predictions" / f"{config_name}_predictions.csv",
        index=False
    )

    save_confusion_matrix(
        y_test,
        test_pred,
        config_name,
        RESULTS_DIR / "confusion_matrices" / f"{config_name}_cm.png"
    )

    return metrics

In [28]:
X = df["path"].values
y = df["label"].values

skf = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=BASE_SEED
)

final_results = []

# Classical models
for classical_model in ["SVM", "RandomForest"]:
    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), start=1):
        result = run_classical_fold(
            df=df,
            train_idx=train_idx,
            test_idx=test_idx,
            fold=fold,
            model_name=classical_model,
            base_seed=BASE_SEED
        )

        final_results.append(result)

        pd.DataFrame(final_results).to_csv(
            RESULTS_DIR / "metrics" / "final_benchmark_partial.csv",
            index=False
        )

# Deep models
for config in FINAL_DEEP_CONFIGS:
    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), start=1):
        result = run_one_fold_config(
            df=df,
            train_idx=train_idx,
            test_idx=test_idx,
            fold=fold,
            config=config,
            base_seed=BASE_SEED
        )

        final_results.append(result)

        pd.DataFrame(final_results).to_csv(
            RESULTS_DIR / "metrics" / "final_benchmark_partial.csv",
            index=False
        )

final_df = pd.DataFrame(final_results)
final_df.to_csv(
    RESULTS_DIR / "metrics" / "final_benchmark_all_results.csv",
    index=False
)

final_df

e:\DL-Projects\Virenvs\py38\lib\site-packages\skimage\feature\texture.py:353: UserWarning: Applying `local_binary_pattern` to floating-point images may give unexpected results when small numerical differences between adjacent pixels are present. It is recommended to use this function with images of integer dtype.
  warnings.warn(
e:\DL-Projects\Virenvs\py38\lib\site-packages\skimage\feature\texture.py:353: UserWarning: Applying `local_binary_pattern` to floating-point images may give unexpected results when small numerical differences between adjacent pixels are present. It is recommended to use this function with images of integer dtype.
  warnings.warn(
e:\DL-Projects\Virenvs\py38\lib\site-packages\skimage\feature\texture.py:353: UserWarning: Applying `local_binary_pattern` to floating-point images may give unexpected results when small numerical differences between adjacent pixels are present. It is recommended to use this function with images of integer dtype.
  warnings.warn(
e:\D


mobilenetv2_focal_lr0.0001_f2_freeze5_fold1
Epoch 001 | Loss 1.0510 | Val F1 0.2857 | Val Recall 1.0000 | Thr 0.472
Epoch 002 | Loss 1.0170 | Val F1 0.3529 | Val Recall 1.0000 | Thr 0.506
Epoch 003 | Loss 0.9382 | Val F1 0.3000 | Val Recall 1.0000 | Thr 0.517
Epoch 004 | Loss 0.6431 | Val F1 0.2609 | Val Recall 1.0000 | Thr 0.519
Epoch 005 | Loss 0.6164 | Val F1 0.2609 | Val Recall 1.0000 | Thr 0.532
Epoch 006 | Loss 0.5390 | Val F1 0.2609 | Val Recall 1.0000 | Thr 0.556
Epoch 007 | Loss 0.4402 | Val F1 0.2609 | Val Recall 1.0000 | Thr 0.567
Epoch 008 | Loss 0.3208 | Val F1 0.2727 | Val Recall 1.0000 | Thr 0.599
Epoch 009 | Loss 0.2788 | Val F1 0.3750 | Val Recall 1.0000 | Thr 0.644
Epoch 010 | Loss 0.2353 | Val F1 0.4286 | Val Recall 1.0000 | Thr 0.670
Epoch 011 | Loss 0.2382 | Val F1 0.2727 | Val Recall 1.0000 | Thr 0.659
Epoch 012 | Loss 0.1743 | Val F1 0.2727 | Val Recall 1.0000 | Thr 0.656
Epoch 013 | Loss 0.1761 | Val F1 0.2727 | Val Recall 1.0000 | Thr 0.664
Epoch 014 | Loss 0.

,accuracy,precision,recall,specificity,f1,balanced_accuracy,mcc,tn,fp,fn,...,pr_auc,fold,model,loss,lr,threshold_mode,threshold,freeze_epochs,use_sampler,config_name
0,0.875000,0.500000,0.500000,0.928571,0.500000,0.714286,0.428571,39,3,3,...,0.675926,1,SVM,classical,NaN,f2,0.629201,NaN,False,SVM_classical_f2_fold1
1,0.729167,0.181818,0.333333,0.785714,0.235294,0.559524,0.093675,33,9,4,...,0.485714,2,SVM,classical,NaN,f2,0.180732,NaN,False,SVM_classical_f2_fold2
2,0.958333,1.000000,0.714286,1.000000,0.833333,0.857143,0.825265,41,0,2,...,0.927644,3,SVM,classical,NaN,f2,0.200161,NaN,False,SVM_classical_f2_fold3
3,0.791667,0.300000,0.500000,0.833333,0.375000,0.666667,0.271448,35,7,3,...,0.425588,1,RandomForest,classical,NaN,f2,0.164000,NaN,False,RandomForest_classical_f2_fold1
4,0.562500,0.222222,1.000000,0.500000,0.363636,0.750000,0.333333,21,21,0,...,0.236485,2,RandomForest,classical,NaN,f2,0.086000,NaN,False,RandomForest_classical_f2_fold2
5,0.687500,0.277778,0.714286,0.682927,0.400000,0.698606,0.289579,28,13,2,...,0.462214,3,RandomForest,classical,NaN,f2,0.120000,NaN,False,RandomForest_classical_f2_fold3
6,0.458333,0.187500,1.000000,0.380952,0.315789,0.690476,0.267261,16,26,0,...,0.727193,1,mobilenetv2,focal,0.0001,f2,0.648807,5.0,True,mobilenetv2_focal_lr0.0001_f2_freeze5_fold1
7,0.479167,0.172414,0.833333,0.428571,0.285714,0.630952,0.177120,18,24,1,...,0.192314,2,mobilenetv2,focal,0.0001,f2,0.671984,5.0,True,mobilenetv2_focal_lr0.0001_f2_freeze5_fold2
8,0.854167,0.500000,0.285714,0.951220,0.363636,0.618467,0.302560,39,2,5,...,0.571303,3,mobilenetv2,focal,0.0001,f2,0.848768,5.0,True,mobilenetv2_focal_lr0.0001_f2_freeze5_fold3
9,0.937500,0.666667,1.000000,0.928571,0.800000,0.964286,0.786796,39,3,0,...,0.976190,1,resnet18,weighted_ce,0.0001,f2,0.802208,5.0,True,resnet18_weighted_ce_lr0.0001_f2_freeze5_fold1


In [29]:
metric_cols = [
    "accuracy", "precision", "recall", "specificity",
    "f1", "balanced_accuracy", "roc_auc", "pr_auc", "mcc"
]

final_df["model_display"] = final_df["model"].replace({
    "RandomForest": "RF",
    "mobilenetv2": "MobileNetV2",
    "resnet18": "ResNet18",
    "resnet34": "ResNet34",
    "densenet121": "DenseNet121"
})

summary_mean = final_df.groupby("model_display")[metric_cols].mean()
summary_std = final_df.groupby("model_display")[metric_cols].std()

paper_table = pd.DataFrame()

for metric in metric_cols:
    paper_table[metric] = (
        summary_mean[metric].map(lambda x: f"{x:.4f}") +
        " ± " +
        summary_std[metric].map(lambda x: f"{x:.4f}")
    )

paper_table = paper_table.reset_index()

paper_table.to_csv(
    RESULTS_DIR / "metrics" / "final_paper_table_mean_std.csv",
    index=False
)

paper_table

,model_display,accuracy,precision,recall,specificity,f1,balanced_accuracy,roc_auc,pr_auc,mcc
0,DenseNet121,0.9583 ± 0.0208,0.8333 ± 0.1443,0.8968 ± 0.0901,0.9679 ± 0.0278,0.8554 ± 0.0546,0.9323 ± 0.0385,0.9454 ± 0.0772,0.9138 ± 0.0759,0.8376 ± 0.0686
1,MobileNetV2,0.5972 ± 0.2228,0.2866 ± 0.1849,0.7063 ± 0.3737,0.5869 ± 0.3164,0.3217 ± 0.0393,0.6466 ± 0.0385,0.7902 ± 0.1324,0.4969 ± 0.2751,0.2490 ± 0.0647
2,RF,0.6806 ± 0.1147,0.2667 ± 0.0401,0.7381 ± 0.2508,0.6721 ± 0.1669,0.3795 ± 0.0186,0.7051 ± 0.0420,0.7814 ± 0.0588,0.3748 ± 0.1211,0.2981 ± 0.0318
3,ResNet18,0.8819 ± 0.0636,0.5463 ± 0.1253,0.8492 ± 0.1435,0.8875 ± 0.0518,0.6643 ± 0.1369,0.8684 ± 0.0963,0.9265 ± 0.0973,0.8159 ± 0.2020,0.6184 ± 0.1720
4,ResNet34,0.9375 ± 0.0417,0.7556 ± 0.2143,0.8968 ± 0.0901,0.9437 ± 0.0505,0.8050 ± 0.1017,0.9202 ± 0.0424,0.9475 ± 0.0358,0.8559 ± 0.0685,0.7830 ± 0.1211
5,SVM,0.8542 ± 0.1160,0.5606 ± 0.4124,0.5159 ± 0.1910,0.9048 ± 0.1091,0.5229 ± 0.2997,0.7103 ± 0.1488,0.8751 ± 0.1173,0.6964 ± 0.2217,0.4492 ± 0.3662


In [30]:
ranking = final_df.groupby("model_display")[
    ["f1", "recall", "precision", "balanced_accuracy", "pr_auc", "mcc"]
].mean()

ranking = ranking.sort_values(
    by=["f1", "recall", "balanced_accuracy"],
    ascending=False
)

ranking.to_csv(
    RESULTS_DIR / "metrics" / "final_model_ranking.csv"
)

ranking

,f1,recall,precision,balanced_accuracy,pr_auc,mcc
model_display,,,,,,
DenseNet121,0.855411,0.896825,0.833333,0.932346,0.913832,0.837632
ResNet34,0.804991,0.896825,0.755556,0.920248,0.855942,0.783037
ResNet18,0.664327,0.849206,0.546296,0.868370,0.815894,0.618412
SVM,0.522876,0.515873,0.560606,0.710317,0.696428,0.449171
RF,0.379545,0.738095,0.266667,0.705091,0.374762,0.298120
MobileNetV2,0.321713,0.706349,0.286638,0.646632,0.496937,0.248980


In [31]:
agg_cm_rows = []

for model_name, group in final_df.groupby("model_display"):
    agg_cm_rows.append({
        "model": model_name,
        "tn": int(group["tn"].sum()),
        "fp": int(group["fp"].sum()),
        "fn": int(group["fn"].sum()),
        "tp": int(group["tp"].sum())
    })

agg_cm_df = pd.DataFrame(agg_cm_rows)

agg_cm_df.to_csv(
    RESULTS_DIR / "metrics" / "final_aggregate_confusion_counts.csv",
    index=False
)

agg_cm_df

,model,tn,fp,fn,tp
0,DenseNet121,121,4,2,17
1,MobileNetV2,73,52,6,13
2,RF,84,41,5,14
3,ResNet18,111,14,3,16
4,ResNet34,118,7,2,17
5,SVM,113,12,9,10


In [32]:
def save_prediction_examples(model_display_name, max_per_group=8):
    pred_files = list((RESULTS_DIR / "predictions").glob("*predictions.csv"))

    selected_files = [
        p for p in pred_files
        if model_display_name.lower().replace(" ", "") in p.name.lower()
    ]

    if len(selected_files) == 0:
        print(f"No prediction files found for {model_display_name}")
        return

    pred_df = pd.concat([pd.read_csv(p) for p in selected_files], ignore_index=True)

    conditions = {
        "TP": (pred_df["y_true"] == 1) & (pred_df["y_pred"] == 1),
        "TN": (pred_df["y_true"] == 0) & (pred_df["y_pred"] == 0),
        "FP": (pred_df["y_true"] == 0) & (pred_df["y_pred"] == 1),
        "FN": (pred_df["y_true"] == 1) & (pred_df["y_pred"] == 0),
    }

    out_dir = RESULTS_DIR / "misclassified" / model_display_name
    out_dir.mkdir(parents=True, exist_ok=True)

    for group_name, condition in conditions.items():
        group_df = pred_df[condition].copy()

        if len(group_df) == 0:
            continue

        group_df = group_df.sort_values("prob_defective", ascending=False)

        for i, row in group_df.head(max_per_group).iterrows():
            img = Image.open(row["path"]).convert("RGB")
            save_name = f"{group_name}_{Path(row['path']).stem}_prob{row['prob_defective']:.3f}.png"
            img.save(out_dir / save_name)

    print(f"Saved examples for {model_display_name} to {out_dir}")


save_prediction_examples("densenet121", max_per_group=8)
save_prediction_examples("resnet34", max_per_group=8)

Saved examples for densenet121 to results_final_benchmark\misclassified\densenet121
Saved examples for resnet34 to results_final_benchmark\misclassified\resnet34


In [33]:
from thop import profile
import copy

def count_params(model):
    return sum(p.numel() for p in model.parameters())


def estimate_model_size_mb(model):
    param_size = 0
    buffer_size = 0

    for param in model.parameters():
        param_size += param.nelement() * param.element_size()

    for buffer in model.buffers():
        buffer_size += buffer.nelement() * buffer.element_size()

    return (param_size + buffer_size) / (1024 ** 2)


def benchmark_inference_time(model, device, img_size=224, runs=100, warmup=20):
    model.eval()
    dummy = torch.randn(1, 3, img_size, img_size).to(device)

    with torch.no_grad():
        for _ in range(warmup):
            _ = model(dummy)

        if device.type == "cuda":
            torch.cuda.synchronize()

        start = time.time()

        for _ in range(runs):
            _ = model(dummy)

        if device.type == "cuda":
            torch.cuda.synchronize()

        end = time.time()

    avg_time = (end - start) / runs
    fps = 1.0 / avg_time

    return avg_time * 1000, fps


def complexity_for_model(model_name):
    model = build_model(model_name).to(device)
    model.eval()

    dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE).to(device)

    try:
        flops, params_thop = profile(
            model,
            inputs=(dummy,),
            verbose=False
        )
    except Exception as e:
        print(f"FLOPs failed for {model_name}: {e}")
        flops = np.nan

    params = count_params(model)
    size_mb = estimate_model_size_mb(model)
    infer_ms, fps = benchmark_inference_time(model, device, IMG_SIZE)

    return {
        "model": model_name,
        "parameters": params,
        "parameters_million": params / 1e6,
        "flops": flops,
        "flops_g": flops / 1e9 if not np.isnan(flops) else np.nan,
        "model_size_mb": size_mb,
        "inference_time_ms": infer_ms,
        "fps": fps
    }


complexity_models = [
    "mobilenetv2",
    "resnet18",
    "resnet34",
    "densenet121"
]

complexity_rows = []

for model_name in complexity_models:
    row = complexity_for_model(model_name)
    complexity_rows.append(row)

complexity_df = pd.DataFrame(complexity_rows)

complexity_df.to_csv(
    RESULTS_DIR / "complexity" / "model_complexity_inference.csv",
    index=False
)

complexity_df

,model,parameters,parameters_million,flops,flops_g,model_size_mb,inference_time_ms,fps
0,mobilenetv2,2226434,2.226434,3.262093e+08,0.326209,8.624504,5.496085,181.947706
1,resnet18,11177538,11.177538,1.823523e+09,1.823523,42.675835,2.379997,420.168636
2,resnet34,21285698,21.285698,3.678228e+09,3.678228,81.263969,4.297574,232.689443
3,densenet121,6955906,6.955906,2.895985e+09,2.895985,26.855698,15.806379,63.265595
